In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.applications import InceptionV3                              
from keras.models import Model
from keras.layers import Dropout,Input,Flatten,Dense,MaxPooling2D       #to give layers
from keras.optimizers import Adam                                       #to optmize the data 
from keras.preprocessing.image import ImageDataGenerator                #to generate the image data 

In [2]:
batchsize=2

In [12]:
tf.test.is_gpu_available()

False

In [4]:
train_datagen=ImageDataGenerator(rescale=1./255, rotation_range=0.2,shear_range=0.2,
                              zoom_range=0.2,width_shift_range=0.2,
                              height_shift_range=0.2, validation_split=0.2)

train_data=train_datagen.flow_from_directory(r'D:\project\project2\Prepared_Data\Train',
                                             target_size=(80,80),batch_size=batchsize,class_mode='categorical',subset='training',color_mode='rgb')
validation_data=train_datagen.flow_from_directory(r'D:\project\project2\Prepared_Data\Train',
                                             target_size=(80,80),batch_size=batchsize,class_mode='categorical',subset='validation',color_mode='rgb')

Found 64530 images belonging to 2 classes.
Found 16132 images belonging to 2 classes.


In [5]:
test_datagen=ImageDataGenerator(rescale=1./255)
test_data=test_datagen.flow_from_directory(r'D:\project\project2\Prepared_Data\Test',
                                             target_size=(80,80),batch_size=batchsize,class_mode='categorical',color_mode='rgb')

Found 4236 images belonging to 2 classes.


In [6]:
base_model=InceptionV3(include_top=False,weights='imagenet',input_tensor=Input(shape=(80,80,3),batch_size=batchsize))
head_model=base_model.output
head_model=Flatten()(head_model)
head_model=Dense(64,activation='relu')(head_model)
head_model=Dropout(0.5)(head_model)
head_model=Dense(2,activation='softmax')(head_model)

model=Model(inputs=base_model.input,outputs=head_model)

for layer in base_model.layers:
    layer.trainable=False

In [7]:
base_model.summary()

Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(2, 80, 80, 3)]             0         []                            
                                                                                                  
 conv2d (Conv2D)             (2, 39, 39, 32)              864       ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (2, 39, 39, 32)              96        ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 activation (Activation)     (2, 39, 39, 32)              0         ['batch_normalizati

In [9]:
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping,ReduceLROnPlateau

In [10]:
checkpoint=ModelCheckpoint(r'D:\project\project2\models',
                           monitor='val_loss', save_best_only=True,verbose=3)

earlystop=EarlyStopping(monitor='val_loss',patience=7,verbose=3,restore_best_weights=True)
learning_rate=ReduceLROnPlateau(monitor='val_loss',patience=3,verbose=3)

callbacks=[checkpoint,earlystop,learning_rate]

In [58]:
model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['accuracy'])

model.fit(train_data,steps_per_epoch=train_data.samples//batchsize,
                    validation_data=validation_data,
                    validation_steps=validation_data.samples//batchsize,
                    callbacks=callbacks,
                    epochs=5)

Epoch 1/5
32265/32265 [==============================] - ETA: 0s - loss: 0.2208 - accuracy: 0.9154
Epoch 1: val_loss improved from inf to 0.19077, saving model to D:\project\project2\models


INFO:tensorflow:Assets written to: D:\project\project2\models\assets


INFO:tensorflow:Assets written to: D:\project\project2\models\assets


32265/32265 [==============================] - 736s 23ms/step - loss: 0.2208 - accuracy: 0.9154 - val_loss: 0.1908 - val_accuracy: 0.9289 - lr: 0.0010
Epoch 2/5
32263/32265 [============================>.] - ETA: 0s - loss: 0.1962 - accuracy: 0.9254
Epoch 2: val_loss did not improve from 0.19077
32265/32265 [==============================] - 670s 21ms/step - loss: 0.1962 - accuracy: 0.9254 - val_loss: 0.2088 - val_accuracy: 0.9136 - lr: 0.0010
Epoch 3/5
32264/32265 [============================>.] - ETA: 0s - loss: 0.1920 - accuracy: 0.9274
Epoch 3: val_loss did not improve from 0.19077
32265/32265 [==============================] - 588s 18ms/step - loss: 0.1920 - accuracy: 0.9274 - val_loss: 0.2028 - val_accuracy: 0.9139 - lr: 0.0010
Epoch 4/5
32262/32265 [============================>.] - ETA: 0s - loss: 0.1895 - accuracy: 0.9296
Epoch 4: val_loss improved from 0.19077 to 0.18504, saving model to D:\project\project2\models


INFO:tensorflow:Assets written to: D:\project\project2\models\assets


INFO:tensorflow:Assets written to: D:\project\project2\models\assets


32265/32265 [==============================] - 603s 19ms/step - loss: 0.1895 - accuracy: 0.9296 - val_loss: 0.1850 - val_accuracy: 0.9262 - lr: 0.0010
Epoch 5/5
32264/32265 [============================>.] - ETA: 0s - loss: 0.1893 - accuracy: 0.9309
Epoch 5: val_loss did not improve from 0.18504
32265/32265 [==============================] - 581s 18ms/step - loss: 0.1893 - accuracy: 0.9309 - val_loss: 0.1944 - val_accuracy: 0.9227 - lr: 0.0010


In [ ]:
acc_tr,loss_tr=model.evaluate_generator(train_data)
print(acc_tr)
print(loss_tr)

In [ ]:
acc_vr,loss_vr=model.evaluate_generator(validation_data)
print(acc_vr)
print(loss_vr)

In [ ]:
acc_test,loss_test=model.evaluate_generator(test_data)
print(acc_tr)
print(loss_tr)

In [ ]:
model.summary()